In [1]:
import pandas as pd
import numpy as np
import csv

In [2]:
#input TableS11 cliques
path = '../../TableS11/TF_cliques_spearman_pearson_08032026.csv'

#turns them into a list of sets
with open(path, newline='') as f:
    cliques = [set(filter(None, (x.strip() for x in row)))
                 for row in csv.reader(f)
                 if any(x.strip() for x in row)]

In [3]:
Fullorthotable = pd.read_csv('../../OTO_star_nothreshold_6species_ohnlogs_expressionthresh_08022026.tsv', delimiter = '\t',index_col = 'MM')

In [11]:
df = pd.read_csv('../../scenic_plus_testing/scenic_plus_guang/eRegulon_direct_guang.tsv', delimiter = '\t')

In [12]:
len(cliques)

323

In [13]:
unique_genes = set.union(*cliques)

In [14]:
len(unique_genes)

226

In [15]:
from itertools import combinations

#Identifies pairs in the cliques that are also coregulatory
fin_set = set()
for item in cliques:
    if len(item) == 2:
        item_l = list(item)
        f1 = item_l[0] in list(df[df['TF'] == item_l[1]]['Gene'])
        f2 = item_l[1] in list(df[df['TF'] == item_l[0]]['Gene'])
        if f1 and f2:
            fin_set.add(tuple(item_l))
    else:
        for a, b in combinations(item, 2):
            f1 = a in list(df[df['TF'] == b]['Gene'])
            f2 = b in list(df[df['TF'] == a]['Gene'])
            if f1 and f2:
                fin_set.add(tuple([a,b]))

In [16]:
len(fin_set)

16

In [17]:
fin_set

{('Dlx1', 'Dlx2'),
 ('Dlx2', 'Dlx1'),
 ('Dlx2', 'Dlx5'),
 ('Dlx2', 'Isl1'),
 ('Dlx5', 'Dlx1'),
 ('Dlx5', 'Dlx2'),
 ('Lhx5', 'Tbr1'),
 ('Lhx5', 'Uncx'),
 ('Nfia', 'Nfix'),
 ('Nfia', 'Tcf4'),
 ('Nfix', 'Tcf4'),
 ('Pax6', 'Meis2'),
 ('Pgr', 'Ar'),
 ('Pgr', 'Esr1'),
 ('Sox1', 'Sox2'),
 ('Sox5', 'Sox2')}

In [10]:
#checking regulation
g1 = 'Pax6'
g2 = 'Meis2'

In [14]:
target = df[df['Gene'] == g1]
target[target['TF'] == g2]

,Region,Gene,importance_R2G,rho_R2G,importance_x_rho,importance_x_abs_rho,TF,is_extended,eRegulon_name,Gene_signature_name,Region_signature_name,importance_TF2G,regulation,rho_TF2G,triplet_rank
20070,chr2:105682059-105682559,Pax6,0.065414,0.595741,0.03897,0.03897,Meis2,False,Meis2_direct_+/+,Meis2_direct_+/+_(28g),Meis2_direct_+/+_(35r),1.026547,1,0.392162,69098


In [15]:
target = df[df['Gene'] == g2]
target[target['TF'] == g1]

,Region,Gene,importance_R2G,rho_R2G,importance_x_rho,importance_x_abs_rho,TF,is_extended,eRegulon_name,Gene_signature_name,Region_signature_name,importance_TF2G,regulation,rho_TF2G,triplet_rank
73895,chr2:116018432-116018932,Meis2,0.011178,-0.140046,-0.001565,0.001565,Pax6,False,Pax6_direct_+/-,Pax6_direct_+/-_(47g),Pax6_direct_+/-_(51r),1.397256,1,0.392162,67998
73903,chr2:116042923-116043423,Meis2,0.008650,-0.158306,-0.001369,0.001369,Pax6,False,Pax6_direct_+/-,Pax6_direct_+/-_(47g),Pax6_direct_+/-_(51r),1.397256,1,0.392162,155269


In [11]:
TF = pd.read_csv('../../Fin_TF_06012026.csv')
TF_set = set(TF['0'])
overall_TF_set = set(Fullorthotable.index) & TF_set

In [12]:
# Build lookup: TF -> set of target genes (using the guang eRegulon)
tf_targets = {}
for tf, grp in df.groupby('TF'):
    tf_targets[tf] = set(grp['Gene'])

# TFs in overall_TF_set that also have a regulon in df
queryable_tfs = TF_set & set(tf_targets.keys())
print(f'TFs in overall_TF_set: {len(overall_TF_set)}')
print(f'TFs with a regulon in eRegulon: {len(tf_targets)}')
print(f'Overlap (queryable): {len(queryable_tfs)}')

TFs in overall_TF_set: 616
TFs with a regulon in eRegulon: 206
Overlap (queryable): 199


In [13]:
from itertools import combinations

# Find all mutually co-regulating TF pairs within overall_TF_set
# A pair (A, B) is co-regulated if A regulates B AND B regulates A
coregulated_pairs = []
for a, b in combinations(sorted(queryable_tfs), 2):
    if b in tf_targets[a] and a in tf_targets[b]:
        coregulated_pairs.append((a, b))

coregulated_tfs = set()
for a, b in coregulated_pairs:
    coregulated_tfs.add(a)
    coregulated_tfs.add(b)

print(f'Co-regulated pairs (mutual regulation): {len(coregulated_pairs)}')
print(f'Unique TFs involved in at least one co-regulatory pair: {len(coregulated_tfs)}')
print()
print('Pairs:')
for a, b in coregulated_pairs:
    print(f'  {a} <-> {b}')

Co-regulated pairs (mutual regulation): 142
Unique TFs involved in at least one co-regulatory pair: 97

Pairs:
  Alx3 <-> Nr2f2
  Ar <-> Pgr
  Arx <-> Dlx1
  Arx <-> Dlx2
  Arx <-> Hmx2
  Arx <-> Hmx3
  Arx <-> Lhx5
  Arx <-> Lhx6
  Ascl1 <-> Ebf1
  Ascl1 <-> Olig2
  Bhlhe22 <-> Neurod1
  Bhlhe22 <-> Neurod2
  Bhlhe22 <-> Pou3f2
  Bsx <-> Nhlh2
  Crem <-> Fosl2
  Cux2 <-> Olig2
  Dbp <-> Rora
  Dlx1 <-> Dlx2
  Dlx1 <-> Dlx5
  Dlx1 <-> Esr1
  Dlx1 <-> Hlf
  Dlx1 <-> Hmx2
  Dlx1 <-> Hmx3
  Dlx1 <-> Lhx6
  Dlx2 <-> Dlx5
  Dlx2 <-> Hmx3
  Dlx2 <-> Isl1
  Dlx2 <-> Tcf7l2
  Ebf1 <-> Neurod1
  Egr3 <-> Ets2
  Elf1 <-> Nfe2l1
  Elf2 <-> Klf13
  Elf4 <-> Fli1
  En2 <-> Klf12
  Esr1 <-> Pgr
  Esrrg <-> Nfib
  Ets2 <-> Nr5a1
  Etv1 <-> Rora
  Etv5 <-> Rfx4
  Fli1 <-> Irf8
  Fli1 <-> Spi1
  Foxo1 <-> Sox9
  Hmx2 <-> Hmx3
  Hmx2 <-> Isl1
  Hmx2 <-> Lhx5
  Hmx3 <-> Isl1
  Hmx3 <-> Lhx5
  Irf5 <-> Irf8
  Irf5 <-> Spi1
  Irf8 <-> Mef2a
  Irf8 <-> Spi1
  Isl1 <-> Uncx
  Jund <-> Nfia
  Jund <-> Rfx4
  

In [13]:
# Summary table: for each TF, how many other TFs in overall_TF_set does it mutually co-regulate with?
from collections import Counter

partner_counts = Counter()
for a, b in coregulated_pairs:
    partner_counts[a] += 1
    partner_counts[b] += 1

summary = pd.DataFrame(
    [(tf, cnt) for tf, cnt in partner_counts.most_common()],
    columns=['TF', 'n_mutual_partners']
)
print(summary.to_string(index=False))

     TF  n_mutual_partners
   Nfia                 11
  Klf12                  9
   Dlx1                  8
   Rora                  8
   Nfib                  8
   Rfx4                  8
   Sox5                  8
   Lhx5                  7
  Klf13                  7
   Nfix                  7
  Tcf12                  7
   Tcf4                  7
    Arx                  6
   Dlx2                  6
   Hmx3                  6
  Prrx1                  6
   Sox6                  6
   Hmx2                  5
 Nfe2l1                  5
   Sox8                  5
 Nkx2-2                  5
   Sox1                  5
  Olig2                  4
Neurod1                  4
   Isl1                  4
   Irf8                  4
   Sox9                  4
  Sox10                  4
   Lhx6                  3
Bhlhe22                  3
Neurod2                  3
   Fli1                  3
  Nr5a1                  3
   Spi1                  3
   Jund                  3
   Pbx1                  3
 